# Impact Platform — Phase 6.5 Forestry Training (Colab)

**Run this notebook in Google Colab with GPU enabled** (Runtime → Change runtime type → GPU)

This trains the **forestry** sapling detector (YOLOv8n) and change detector (ChangeFormer).

Expected time: ~2-3 hours on T4

In [ ]:
# @title 1. Bootstrap environment
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_URL = "https://github.com/YOUR_ORG/impact-platform.git"  # <-- CHANGE THIS
REPO_DIR = "/content/drive/MyDrive/impact-platform"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo exists, pulling latest...")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(os.path.join(REPO_DIR, "apps", "ml-service"))
print(f"Working in: {os.getcwd()}")

In [ ]:
# @title 2. Install requirements
!pip install -q -r training/requirements_colab.txt

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# @title 3. Download datasets (ForestNet + LEVIR-CD)
!python -m training.data.download_datasets download_forestnet
!python -m training.data.download_datasets download_levir_cd

In [ ]:
# @title 4. Prepare YOLO format
!python -m training.data.prepare_yolo forestry

In [ ]:
# @title 5. Train YOLOv8n sapling detector (~2 hrs on T4)
!python -m training.scripts.train_yolo forestry

In [ ]:
# @title 6. Train ChangeFormer on LEVIR-CD (~1 hr on T4)
!python -m training.scripts.train_change forestry

In [ ]:
# @title 7. Evaluate
!python -m training.scripts.evaluate forestry --output eval/forestry_eval_v1.0.json

In [ ]:
# @title 8. Verify weights exist
from pathlib import Path
weights_dir = Path("weights")
for w in ["sapling_yolov8n.pt", "changeformer.pt", "yolov8n.pt"]:
    p = weights_dir / w
    print(f"{w}: {'EXISTS' if p.exists() else 'MISSING'} ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# @title 9. (Optional) Export to ONNX
# !python -m training.scripts.export_onnx forestry

In [ ]:
# @title 10. Promote model registry (run AFTER uploading weights to bucket)
# This requires SUPABASE_SERVICE_KEY and SUPABASE_URL env vars
# !python -m training.scripts.promote_model forestry --version v1.0